In [49]:
#!pip install -U sentence-transformers
#!pip install faiss-cpu
#!pip install faiss-gpu
#!pip install numpy requests

from sentence_transformers import SentenceTransformer
import requests
import faiss
import os

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [11]:
import numpy as np

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [78]:
test_dict = [
    {"content": "You can book a doctor's appointment through the number online"},
    {"content": "A doctor's appointment can be booked online using the available contact number"},
    {"content": "You can arrange a medical appointment by calling the number listed online"},
    {"content": "If you want a dog, you'll need to get your allowance"},
    {"content": "This is a very interesting book"},
    {"content": "eafojpeafoiheafoeahiofehofeahfiafoh"}]

# Similarity threshold for cosine similiarity
similarity_threshold = 0.8;

# Create embedding function which will receive multiple dictionaries and build embeddings from the text value of the dictionaries
def create_embeddings(chunks):

    # Content provides the text value of the chunk
    content = []
    
    for c in chunks:
        
        content.append(c["content"])
        
    # Create vector embeddings for content
    embeddings = model.encode(content, batch_size=32, normalize_embeddings = True)

    # Return vector embeddings which can be stored in vectordb
    return embeddings.astype("float32")


embeddings = create_embeddings(test_dict)

# Get embedding shape for Faiss sotrage
dimension = embeddings.shape[1]

# IndexFlatL2 for simple and exact- we might need to change it to approximate nearest neighbour to improve on peroformance
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(embeddings)



[[ 0.03116288 -0.01431826  0.01073793 ... -0.01685228 -0.01724875
  -0.00910183]
 [-0.03481282  0.0233593  -0.02919483 ...  0.02535159 -0.03104476
   0.00380932]
 [ 0.03022364 -0.0350099   0.0100458  ... -0.05111761 -0.01299704
  -0.02129326]
 [-0.02024278 -0.01845082  0.05717922 ...  0.03762465  0.00794937
  -0.01940895]
 [-0.02787443  0.02362619  0.0130731  ...  0.04500193 -0.08570333
  -0.00612703]
 [ 0.08688255  0.06724466 -0.00317573 ...  0.05716873  0.12446596
  -0.00146962]]


In [90]:
# Test question
query = "medical book"
query_embedding = model.encode([query], normalize_embeddings = True)

D, I = index.search(query_embedding, k=3)
print("Distances:", D)
print("Indices:", I)

for i in range(3):
    indice = I[0][i]
    distance = D[0][i]
    print(test_dict[indice]["content"])

Distances: [[1.1513959 1.2114842 1.2230065]]
Indices: [[4 2 0]]
This is a very interesting book
You can arrange a medical appointment by calling the number listed online
You can book a doctor's appointment through the number online
